# Surrogates

A surrogate learns a model's outputs from a dataset of solves; the solver stays the reference. See [Sensitivities and inference](/OpenCR/guides/inference/#surrogates).

In [ ]:
import numpy as np

import opencr
from opencr.inference import Observation, fit
from opencr.ml import Surrogate, generate_dataset

model = opencr.load_model("hydrogen/atomic/basic", n_max=10)
conditions = opencr.Conditions(5.0, 1e18, densities={"H(n=1)": 1e18, "H+": 1e18})
outputs = [
    "effective_rate:H(n=1)>H+",
    "effective_rate:H+>H(n=1)",
    "electron_cooling:H(n=1)",
]
bounds = {"electron_temperature": (1.0, 50.0), "electron_density": (1e17, 1e20)}
dataset = generate_dataset(model, conditions, bounds, outputs, samples=200, seed=1)
print({k: dataset.attrs[k] for k in ("model", "manifest_sha256", "seed", "solve_time")})

## Held-out errors and speed-up

In [ ]:
surrogate = Surrogate().fit(dataset)
report = surrogate.evaluate(dataset)
print(report.to_dataframe())
print(f"speed-up {report.attrs['speed_up']:.0f}x")

## Out of domain

Outside the training ranges the surrogate is not to be trusted, and says so:

In [ ]:
prediction = surrogate.predict([[5.0, 1e18], [200.0, 1e18]])
print(prediction.in_domain)

## A fast forward model

The surrogate's `forward` plugs into the inverse modelling in place of the solver:

In [ ]:
truth = surrogate.forward(np.array([4.0, 2e18]))
observations = [
    Observation(n, float(v), 0.05) for n, v in zip(outputs, truth, strict=True)
]
print(fit(surrogate.forward, observations, bounds).parameters)